# 🛡️ Entrenamiento Avanzado — Detección de Vulnerabilidades en Código Fuente

## Implementación Completa de las 3 Técnicas de `teoria.md`

Este notebook implementa un pipeline de entrenamiento avanzado que aplica **todas** las estrategias de optimización descritas en `teoria.md`:

1. **Weighted Cross-Entropy (WCE)** — Pesos fijos basados en proporción inversa de clases
2. **Focal Loss (FL)** — Factor de modulación para muestras difíciles (γ ∈ [2.0, 3.0])
3. **Class-Balanced Loss (CB)** — Número efectivo de muestras con β = (N-1)/N

### Datasets Utilizados
- MegaVul C/C++ y Java
- CodeXGLUE (train + test + validation)
- D2A IBM (train + dev + test)
- ReVeal Chromium/Debian (train + test + validation)
- CVEFixes CSV
- OWASP 2025

### Modelos Evaluados
- Random Forest
- XGBoost
- LightGBM

---

## Etapa 1: Carga y Exploración de Datos

In [1]:
# === IMPORTS GLOBALES ===
import sys
import os
import json
import re
import warnings
import time
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for notebooks
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# Agregar project root al path
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Directorio del proyecto: {PROJECT_ROOT}")
print(f"✅ Imports completados exitosamente")

📁 Directorio del proyecto: C:\Users\gamur\Documents\ESPE VII SI 2026\Desarrollo Seguro\U2\p\ProyectoSegundoParcialSWSeguro
✅ Imports completados exitosamente


In [2]:
# === FUNCIONES DE CARGA DE DATASETS ===

def load_jsonl(path, code_field='func', label_field='target', limit=None):
    """Carga un archivo JSONL genérico."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            try:
                item = json.loads(line.strip())
                code = item.get(code_field, '') or ''
                # Try multiple field names
                if not code:
                    for alt in ['func', 'code', 'functionSource', 'raw_code']:
                        code = item.get(alt, '')
                        if code:
                            break
                label = item.get(label_field, 0)
                if not label:
                    for alt in ['target', 'label', 'is_vulnerable']:
                        label = item.get(alt, 0)
                        if label:
                            break
                if code and len(code) > 10:
                    data.append({
                        'raw_code': code,
                        'is_vulnerable': int(label)
                    })
            except json.JSONDecodeError:
                continue
    return data

def load_megavul(path, limit=None):
    """Carga MegaVul JSON (formato de pares vuln/safe)."""
    import ijson
    data = []
    count = 0
    with open(path, 'rb') as f:
        for item in ijson.items(f, 'item'):
            if limit and count >= limit:
                break
            # Versión vulnerable (antes del parche)
            func_before = item.get('func_before', '')
            if func_before and len(func_before) > 10:
                data.append({'raw_code': func_before, 'is_vulnerable': 1})
            # Versión parcheada (segura)
            func_after = item.get('func', '')
            if func_after and len(func_after) > 10:
                data.append({'raw_code': func_after, 'is_vulnerable': 0})
            count += 1
    return data

def load_cvefixes_csv(path, limit=None):
    """Carga CVEFixes desde CSV."""
    df = pd.read_csv(path, nrows=limit)
    df = df.dropna(subset=['code', 'safety'])
    df['is_vulnerable'] = df['safety'].apply(lambda x: 1 if x == 'vulnerable' else 0)
    df['raw_code'] = df['code']
    records = df[['raw_code', 'is_vulnerable']].to_dict(orient='records')
    # Filtrar muestras con código muy corto
    return [r for r in records if len(r['raw_code']) > 10]

print("✅ Funciones de carga definidas")

✅ Funciones de carga definidas


In [3]:
# === CARGAR TODOS LOS DATASETS ===
# Para la prueba inicial: limitamos muestras por fuente
# Al final del notebook se re-entrena con TODO

PRUEBA_LIMIT = 5000  # Muestras por fuente para prueba rápida
all_datasets = {}  # nombre -> lista de muestras

# --- 1. CodeXGLUE (train + test + validation) ---
print("\n📦 Cargando CodeXGLUE...")
codexglue_files = [
    ('codexglue_train', 'data/codexglue/train.jsonl'),
    ('codexglue_test', 'data/codexglue/test.jsonl'),
    ('codexglue_valid', 'data/codexglue/validation.jsonl'),
]
for name, path in codexglue_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='func', label_field='target', limit=PRUEBA_LIMIT)
        all_datasets[name] = data
        vuln = sum(1 for d in data if d['is_vulnerable'])
        print(f"  ✅ {name}: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
    else:
        print(f"  ⚠️ {name}: archivo no encontrado ({path})")

# --- 2. D2A IBM (train + dev + test) ---
print("\n📦 Cargando D2A (IBM)...")
d2a_files = [
    ('d2a_train', 'data/d2a/train.jsonl'),
    ('d2a_dev', 'data/d2a/dev.jsonl'),
    ('d2a_test', 'data/d2a/test.jsonl'),
]
for name, path in d2a_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='functionSource', label_field='label', limit=PRUEBA_LIMIT)
        all_datasets[name] = data
        vuln = sum(1 for d in data if d['is_vulnerable'])
        print(f"  ✅ {name}: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
    else:
        print(f"  ⚠️ {name}: archivo no encontrado ({path})")

# --- 3. ReVeal (train + test + validation) ---
print("\n📦 Cargando ReVeal (Chromium/Debian)...")
reveal_files = [
    ('reveal_train', 'data/reveal/train.jsonl'),
    ('reveal_test', 'data/reveal/test.jsonl'),
    ('reveal_valid', 'data/reveal/validation.jsonl'),
]
for name, path in reveal_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='code', label_field='label', limit=PRUEBA_LIMIT)
        all_datasets[name] = data
        vuln = sum(1 for d in data if d['is_vulnerable'])
        print(f"  ✅ {name}: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
    else:
        print(f"  ⚠️ {name}: archivo no encontrado ({path})")

# --- 4. MegaVul C/C++ ---
print("\n📦 Cargando MegaVul C/C++... (esto puede tomar unos segundos)")
megavul_cpp_path = PROJECT_ROOT / 'data/megavul/c_cpp/megavul_simple.json'
if megavul_cpp_path.exists():
    data = load_megavul(megavul_cpp_path, limit=PRUEBA_LIMIT)
    all_datasets['megavul_cpp'] = data
    vuln = sum(1 for d in data if d['is_vulnerable'])
    print(f"  ✅ megavul_cpp: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
else:
    print(f"  ⚠️ megavul_cpp: archivo no encontrado")

# --- 5. MegaVul Java ---
print("\n📦 Cargando MegaVul Java... (esto puede tomar unos segundos)")
megavul_java_path = PROJECT_ROOT / 'data/megavul/java/megavul_simple.json'
if megavul_java_path.exists():
    data = load_megavul(megavul_java_path, limit=PRUEBA_LIMIT)
    all_datasets['megavul_java'] = data
    vuln = sum(1 for d in data if d['is_vulnerable'])
    print(f"  ✅ megavul_java: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
else:
    print(f"  ⚠️ megavul_java: archivo no encontrado")

# --- 6. CVEFixes CSV ---
print("\n📦 Cargando CVEFixes CSV...")
cvefixes_path = PROJECT_ROOT / 'data/CVEFixes.csv/CVEFixes.csv'
if cvefixes_path.exists():
    data = load_cvefixes_csv(cvefixes_path, limit=PRUEBA_LIMIT * 2)  # Extra por el filtrado
    all_datasets['cvefixes'] = data[:PRUEBA_LIMIT]
    vuln = sum(1 for d in all_datasets['cvefixes'] if d['is_vulnerable'])
    print(f"  ✅ cvefixes: {len(all_datasets['cvefixes'])} muestras (vuln={vuln}, safe={len(all_datasets['cvefixes'])-vuln})")
else:
    print(f"  ⚠️ cvefixes: archivo no encontrado")

# --- 7. OWASP 2025 ---
print("\n📦 Cargando OWASP 2025...")
owasp_files = [
    ('owasp_train', 'data/owasp2025/train.jsonl'),
    ('owasp_test', 'data/owasp2025/test.jsonl'),
]
for name, path in owasp_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='raw_code', label_field='is_vulnerable')
        all_datasets[name] = data
        vuln = sum(1 for d in data if d['is_vulnerable'])
        print(f"  ✅ {name}: {len(data)} muestras (vuln={vuln}, safe={len(data)-vuln})")
    else:
        print(f"  ⚠️ {name}: archivo no encontrado ({path})")

print("\n" + "="*70)
print(f"📊 TOTAL DATASETS CARGADOS: {len(all_datasets)}")
total_samples = sum(len(v) for v in all_datasets.values())
total_vuln = sum(sum(1 for d in v if d['is_vulnerable']) for v in all_datasets.values())
print(f"📊 TOTAL MUESTRAS: {total_samples:,}")
print(f"📊 VULNERABLES: {total_vuln:,} ({100*total_vuln/total_samples:.1f}%)")
print(f"📊 SEGURAS: {total_samples - total_vuln:,} ({100*(total_samples-total_vuln)/total_samples:.1f}%)")
print("="*70)


📦 Cargando CodeXGLUE...
  ✅ codexglue_train: 5000 muestras (vuln=2322, safe=2678)
  ✅ codexglue_test: 2732 muestras (vuln=1255, safe=1477)
  ✅ codexglue_valid: 2732 muestras (vuln=1187, safe=1545)

📦 Cargando D2A (IBM)...
  ✅ d2a_train: 4643 muestras (vuln=2487, safe=2156)
  ✅ d2a_dev: 596 muestras (vuln=308, safe=288)
  ✅ d2a_test: 618 muestras (vuln=618, safe=0)

📦 Cargando ReVeal (Chromium/Debian)...
  ✅ reveal_train: 5000 muestras (vuln=476, safe=4524)
  ✅ reveal_test: 2274 muestras (vuln=230, safe=2044)
  ✅ reveal_valid: 2273 muestras (vuln=210, safe=2063)

📦 Cargando MegaVul C/C++... (esto puede tomar unos segundos)


ModuleNotFoundError: No module named 'ijson'

In [ ]:
# === VISUALIZACIÓN DEL BALANCE DE CLASES POR FUENTE ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Distribución de muestras por fuente
sources = list(all_datasets.keys())
vuln_counts = [sum(1 for d in all_datasets[s] if d['is_vulnerable']) for s in sources]
safe_counts = [sum(1 for d in all_datasets[s] if not d['is_vulnerable']) for s in sources]

x = np.arange(len(sources))
width = 0.35
bars1 = axes[0].bar(x - width/2, vuln_counts, width, label='Vulnerable', color='#e74c3c', alpha=0.85)
bars2 = axes[0].bar(x + width/2, safe_counts, width, label='Seguro', color='#2ecc71', alpha=0.85)
axes[0].set_xlabel('Fuente del Dataset')
axes[0].set_ylabel('Cantidad de Muestras')
axes[0].set_title('Distribución de Clases por Fuente')
axes[0].set_xticks(x)
axes[0].set_xticklabels(sources, rotation=45, ha='right', fontsize=8)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico 2: Balance global
total_v = sum(vuln_counts)
total_s = sum(safe_counts)
axes[1].pie(
    [total_v, total_s],
    labels=[f'Vulnerable\n{total_v:,} ({100*total_v/(total_v+total_s):.1f}%)',
            f'Seguro\n{total_s:,} ({100*total_s/(total_v+total_s):.1f}%)'],
    colors=['#e74c3c', '#2ecc71'],
    autopct='',
    startangle=90,
    explode=(0.05, 0)
)
axes[1].set_title('Balance Global de Clases')

plt.tight_layout()
plt.savefig('reports/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado en reports/class_distribution.png")

In [ ]:
# === COMBINAR TODOS LOS DATASETS ===

combined_data = []
for source_name, samples in all_datasets.items():
    for s in samples:
        s_copy = s.copy()
        s_copy['source'] = source_name
        combined_data.append(s_copy)

# Shuffle
random.shuffle(combined_data)

print(f"✅ Dataset combinado: {len(combined_data):,} muestras")

# Crear DataFrame
df_combined = pd.DataFrame(combined_data)
print(f"\n📊 Resumen del DataFrame:")
print(df_combined.info())
print(f"\n📊 Distribución de clases:")
print(df_combined['is_vulnerable'].value_counts())
print(f"\n📊 Distribución por fuente:")
print(df_combined['source'].value_counts())

## Etapa 2: Preprocesamiento y Feature Engineering Mejorado

In [ ]:
# === FEATURE ENGINEERING: SECURITY-AWARE FEATURES ===

class SecurityFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extrae features de seguridad específicas del código fuente.
    Detecta patrones comunes de vulnerabilidades conocidas.
    """
    
    # Funciones peligrosas por categoría
    DANGEROUS_FUNCTIONS = {
        'buffer_overflow': [
            r'\bstrcpy\s*\(', r'\bstrcat\s*\(', r'\bsprintf\s*\(',
            r'\bgets\s*\(', r'\bscanf\s*\(', r'\bvsprintf\s*\(',
            r'\bmemcpy\s*\(', r'\bmemmove\s*\(',
        ],
        'format_string': [
            r'\bprintf\s*\([^"]*\bvar', r'\bfprintf\s*\([^,]+,[^"]*\bvar',
            r'\bsyslog\s*\([^,]+,[^"]*\bvar',
        ],
        'injection': [
            r'\bexec\s*\(', r'\beval\s*\(', r'\bsystem\s*\(',
            r'\bpopen\s*\(', r'\bexecve\s*\(',
            r'SELECT.*FROM.*WHERE', r'INSERT\s+INTO',
            r'DELETE\s+FROM', r'UPDATE.*SET',
            r'\$_GET', r'\$_POST', r'\$_REQUEST',
        ],
        'memory_mgmt': [
            r'\bmalloc\s*\(', r'\bcalloc\s*\(', r'\brealloc\s*\(',
            r'\bfree\s*\(', r'\bdelete\s+', r'\bdelete\[\]',
            r'\bnew\s+\w+',
        ],
        'crypto_weak': [
            r'\bMD5\b', r'\bSHA1\b', r'\bDES\b', r'\bRC4\b',
            r'\brand\s*\(', r'\bsrand\s*\(',
        ],
        'xss': [
            r'innerHTML', r'document\.write', r'\$\.html\s*\(',
            r'dangerouslySetInnerHTML',
        ],
        'auth_issues': [
            r'password', r'secret', r'token', r'api_key',
            r'hardcoded', r'admin',
        ],
    }
    
    SAFE_PATTERNS = [
        r'\bstrncpy\s*\(', r'\bsnprintf\s*\(', r'\bstrlcpy\s*\(',
        r'\bfgets\s*\(', r'\bbounds_check',
        r'\bsizeof\s*\(', r'\bstrncat\s*\(',
        r'\bprepared_statement', r'\bparameterized',
        r'\bsanitize', r'\bescape', r'\bvalidate',
        r'\bif\s*\(.*\bNULL\b', r'\bif\s*\(.*\bnullptr\b',
        r'\bif\s*\(.*\blen\b',
    ]
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        features = []
        for code in X:
            if not isinstance(code, str):
                code = ''
            row = []
            
            # Conteo por categoría de funciones peligrosas
            for category, patterns in self.DANGEROUS_FUNCTIONS.items():
                count = sum(len(re.findall(p, code, re.IGNORECASE)) for p in patterns)
                row.append(count)
            
            # Conteo de patrones seguros
            safe_count = sum(len(re.findall(p, code, re.IGNORECASE)) for p in self.SAFE_PATTERNS)
            row.append(safe_count)
            
            # Ratio de seguridad (safe / (dangerous + 1))
            total_dangerous = sum(row[:-1])
            row.append(safe_count / (total_dangerous + 1))
            
            # Métricas de complejidad
            lines = code.split('\n')
            row.append(len(code))  # longitud del código
            row.append(len(lines))  # número de líneas
            
            # Complejidad ciclomática simplificada
            branches = len(re.findall(r'\b(if|else|elif|while|for|switch|case|catch|&&|\|\|)\b', code))
            row.append(branches)
            
            # Profundidad de anidamiento (estimación por indentación)
            max_indent = 0
            for line in lines:
                stripped = line.lstrip()
                if stripped:
                    indent = len(line) - len(stripped)
                    max_indent = max(max_indent, indent)
            row.append(max_indent)
            
            # Ratio de comentarios
            comment_lines = sum(1 for l in lines if l.strip().startswith(('//', '#', '/*', '*')))
            row.append(comment_lines / (len(lines) + 1))
            
            # Conteo de punteros/referencias (C/C++)
            pointer_count = len(re.findall(r'[*&]\w+|->|\w+\[\d+\]', code))
            row.append(pointer_count)
            
            # Conteo de casts (potencial type confusion)
            cast_count = len(re.findall(r'\(\s*(int|char|void|long|unsigned|float|double)\s*\*?\s*\)', code))
            row.append(cast_count)
            
            features.append(row)
        
        from scipy.sparse import csr_matrix
        return csr_matrix(np.array(features, dtype=np.float64))
    
    def get_feature_names_out(self, input_features=None):
        names = [f'sec_{cat}' for cat in self.DANGEROUS_FUNCTIONS.keys()]
        names += [
            'sec_safe_patterns', 'sec_safety_ratio',
            'code_length', 'code_lines',
            'cyclomatic_complexity', 'max_indent_depth',
            'comment_ratio', 'pointer_count', 'cast_count'
        ]
        return np.array(names)

print("✅ SecurityFeatureExtractor definido")
print(f"   Categorías de funciones peligrosas: {list(SecurityFeatureExtractor.DANGEROUS_FUNCTIONS.keys())}")
print(f"   Features de seguridad totales: {len(SecurityFeatureExtractor.DANGEROUS_FUNCTIONS) + 9}")

In [ ]:
# === AST FEATURE EXTRACTOR ===
# Reutilizamos el del proyecto pero robusto para múltiples lenguajes

class RobustASTFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extrae features del AST con soporte robusto para C/C++ y Java."""
    
    def __init__(self):
        self._cpp_parser = None
        self._java_parser = None
    
    def _init_parsers(self):
        if self._cpp_parser is None:
            try:
                import tree_sitter
                import tree_sitter_cpp
                import tree_sitter_java
                
                cpp_lang = tree_sitter.Language(tree_sitter_cpp.language(), 'cpp')
                java_lang = tree_sitter.Language(tree_sitter_java.language(), 'java')
                
                self._cpp_parser = tree_sitter.Parser()
                self._cpp_parser.set_language(cpp_lang)
                self._java_parser = tree_sitter.Parser()
                self._java_parser.set_language(java_lang)
            except Exception as e:
                print(f"⚠️ Tree-sitter no disponible: {e}. Usando features basadas en regex.")
                self._cpp_parser = 'unavailable'
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        self._init_parsers()
        features = []
        
        for code in X:
            if not isinstance(code, str) or not code.strip():
                features.append([0, 0, 0, 0, 0, 0])
                continue
            
            if self._cpp_parser == 'unavailable':
                # Fallback: regex-based estimation
                features.append(self._regex_features(code))
            else:
                try:
                    # Detect language
                    if re.search(r'\b(public\s+class|import\s+java)\b', code):
                        parser = self._java_parser
                    else:
                        parser = self._cpp_parser
                    
                    tree = parser.parse(bytes(code, 'utf8'))
                    stats = self._extract_stats(tree.root_node)
                    features.append([
                        stats['node_count'],
                        stats['max_depth'],
                        stats['pointer_ops'],
                        stats['function_calls'],
                        stats['loops'],
                        stats['if_statements']
                    ])
                except Exception:
                    features.append(self._regex_features(code))
        
        from scipy.sparse import csr_matrix
        return csr_matrix(np.array(features))
    
    def _regex_features(self, code):
        """Fallback: estimar features AST con regex."""
        node_count = len(code.split())
        max_depth = max((len(l) - len(l.lstrip()) for l in code.split('\n') if l.strip()), default=0)
        pointer_ops = len(re.findall(r'[*&]\w+|->', code))
        function_calls = len(re.findall(r'\w+\s*\(', code))
        loops = len(re.findall(r'\b(for|while|do)\b', code))
        if_stmts = len(re.findall(r'\bif\b', code))
        return [node_count, max_depth, pointer_ops, function_calls, loops, if_stmts]
    
    def _extract_stats(self, root_node):
        stats = {'node_count': 0, 'max_depth': 0, 'pointer_ops': 0,
                 'function_calls': 0, 'loops': 0, 'if_statements': 0}
        stack = [(root_node, 0)]
        while stack:
            node, depth = stack.pop()
            stats['node_count'] += 1
            stats['max_depth'] = max(stats['max_depth'], depth)
            node_type = node.type
            if node_type in ['pointer_declarator', 'reference_declarator', 'pointer_expression']:
                stats['pointer_ops'] += 1
            if node_type in ['call_expression', 'method_invocation']:
                stats['function_calls'] += 1
            if node_type in ['while_statement', 'for_statement', 'do_statement']:
                stats['loops'] += 1
            if node_type in ['if_statement']:
                stats['if_statements'] += 1
            for child in node.children:
                stack.append((child, depth + 1))
        return stats
    
    def get_feature_names_out(self, input_features=None):
        return np.array([
            'ast_node_count', 'ast_max_depth', 'ast_pointer_ops',
            'ast_function_calls', 'ast_loops', 'ast_if_statements'
        ])

print("✅ RobustASTFeatureExtractor definido")

In [ ]:
# === LANGUAGE DETECTOR FEATURE ===

class LanguageFeatureExtractor(BaseEstimator, TransformerMixin):
    """Detecta lenguaje de programación y genera one-hot encoding."""
    
    LANGUAGE_PATTERNS = {
        'java': [r'public\s+class\s+\w+', r'import\s+java\.', r'@Override', r'System\.(out|in|err)'],
        'c': [r'#include\s*<.*\.h>', r'printf\s*\(', r'int\s+main\s*\('],
        'cpp': [r'#include\s*<.*>', r'std::', r'cout\s*<<', r'namespace\s+\w+', r'template\s*<'],
        'python': [r'def\s+\w+\s*\(', r'from\s+\w+\s+import', r'if\s+__name__\s*=='],
        'javascript': [r'function\s+\w+\s*\(', r'const\s+\w+\s*=', r'let\s+\w+\s*=', r'console\.(log|error)'],
        'php': [r'<\?php', r'\$\w+\s*=', r'echo\s+'],
    }
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        features = []
        for code in X:
            if not isinstance(code, str):
                code = ''
            scores = {}
            for lang, patterns in self.LANGUAGE_PATTERNS.items():
                scores[lang] = sum(1 for p in patterns if re.search(p, code, re.MULTILINE))
            max_score = max(scores.values()) if scores else 0
            detected = 'unknown'
            if max_score > 0:
                detected = max(scores, key=scores.get)
            
            row = [1 if detected == lang else 0 for lang in self.LANGUAGE_PATTERNS.keys()]
            row.append(1 if detected == 'unknown' else 0)  # lang_other
            features.append(row)
        from scipy.sparse import csr_matrix
        return csr_matrix(np.array(features))
    
    def get_feature_names_out(self, input_features=None):
        names = [f'lang_{lang}' for lang in self.LANGUAGE_PATTERNS.keys()]
        names.append('lang_other')
        return np.array(names)

print("✅ LanguageFeatureExtractor definido")

In [ ]:
# === PREPARAR DATOS PARA ENTRENAMIENTO ===

# Extraer X (código) y y (etiquetas)
X_raw = df_combined['raw_code'].values
y = df_combined['is_vulnerable'].values

print(f"📊 Datos preparados:")
print(f"   X_raw shape: {X_raw.shape}")
print(f"   y shape: {y.shape}")
print(f"   Clase 0 (seguro): {np.sum(y == 0):,}")
print(f"   Clase 1 (vulnerable): {np.sum(y == 1):,}")

# === SPLIT TRAIN/TEST CON ESTRATIFICACIÓN ===
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Split train/test:")
print(f"   Train: {len(X_train_raw):,} muestras (vuln={np.sum(y_train):,}, safe={np.sum(y_train==0):,})")
print(f"   Test:  {len(X_test_raw):,} muestras (vuln={np.sum(y_test):,}, safe={np.sum(y_test==0):,})")

In [ ]:
# === CONSTRUIR PIPELINE DE FEATURES MEJORADO ===

print("🔧 Construyendo pipeline de features mejorado...")
start_time = time.time()

feature_pipeline = FeatureUnion([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 3),
        sublinear_tf=True,
        min_df=2,
        max_df=0.95,
        strip_accents='unicode',
        token_pattern=r'\b\w+\b'
    )),
    ('ast', RobustASTFeatureExtractor()),
    ('security', SecurityFeatureExtractor()),
    ('language', LanguageFeatureExtractor()),
])

# Fit y transform sobre train
print("   Fitting feature pipeline en datos de entrenamiento...")
X_train_features = feature_pipeline.fit_transform(X_train_raw)
print(f"   ✅ X_train shape: {X_train_features.shape}")

# Transform sobre test
print("   Transformando datos de test...")
X_test_features = feature_pipeline.transform(X_test_raw)
print(f"   ✅ X_test shape: {X_test_features.shape}")

elapsed = time.time() - start_time
print(f"\n⏱️ Feature engineering completado en {elapsed:.1f} segundos")

# Nombres de features
feature_names = feature_pipeline.get_feature_names_out()
print(f"📊 Total de features: {len(feature_names)}")

## Etapa 3: Entrenamiento con Weighted Cross-Entropy (WCE)

**teoria.md**: *"Modifica la entropía cruzada estándar asignando pesos fijos y diferenciados a cada clase, penalizando de forma más severa los errores en la clase minoritaria (vulnerable)."*

$$\mathcal{L}_{\text{WCE}} = - \frac{1}{N} \sum_{i=1}^{N} \left[ w_1 y_i \log(\hat{y}_i) + w_0 (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Para modelos sklearn: `class_weight` y/o `scale_pos_weight` calculados como inversas de las proporciones de clases.

In [ ]:
# === CALCULAR PESOS WCE ===

n_vuln = np.sum(y_train == 1)
n_safe = np.sum(y_train == 0)
n_total = len(y_train)

# Pesos inversamente proporcionales a la frecuencia de clase
w1 = n_total / (2 * n_vuln)  # Peso para clase vulnerable
w0 = n_total / (2 * n_safe)  # Peso para clase segura

# Ratio para XGBoost
scale_pos_weight = n_safe / n_vuln

print("📊 Pesos WCE calculados:")
print(f"   w1 (vulnerable): {w1:.4f}")
print(f"   w0 (seguro):     {w0:.4f}")
print(f"   scale_pos_weight (XGBoost): {scale_pos_weight:.4f}")
print(f"   Ratio vuln/safe: {n_vuln/n_safe:.4f}")
print(f"   n_vuln={n_vuln}, n_safe={n_safe}")

In [ ]:
# === FUNCIÓN COMÚN DE EVALUACIÓN ===

def evaluate_model(model, X_test, y_test, model_name=''):
    """Evalúa un modelo y devuelve métricas detalladas."""
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'model': model_name,
        'accuracy': round(accuracy_score(y_test, predictions), 4),
        'precision': round(precision_score(y_test, predictions, zero_division=0), 4),
        'recall': round(recall_score(y_test, predictions, zero_division=0), 4),
        'f1_score': round(f1_score(y_test, predictions, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_test, probabilities), 4),
        'confusion_matrix': confusion_matrix(y_test, predictions).tolist(),
    }
    
    cm = metrics['confusion_matrix']
    print(f"\n{'='*60}")
    print(f"  📊 RESULTADOS: {model_name}")
    print(f"{'='*60}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1_score']:.4f}")
    print(f"  ROC AUC:   {metrics['roc_auc']:.4f}")
    print(f"  Confusion Matrix: TN={cm[0][0]}, FP={cm[0][1]}, FN={cm[1][0]}, TP={cm[1][1]}")
    print(f"{'='*60}")
    
    return metrics

# Almacén global de resultados para comparación
all_results = []
print("✅ Función de evaluación definida")

In [ ]:
# === WCE: RANDOM FOREST ===

print("\n🌲 Entrenando Random Forest con WCE (class_weight ajustado)...")
start_time = time.time()

rf_wce = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight={0: w0, 1: w1},
    max_features='sqrt',
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
rf_wce.fit(X_train_features, y_train)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_rf_wce = evaluate_model(rf_wce, X_test_features, y_test, 'RandomForest + WCE')
metrics_rf_wce['loss_type'] = 'WCE'
metrics_rf_wce['train_time'] = round(elapsed, 2)
all_results.append(metrics_rf_wce)

In [ ]:
# === WCE: XGBOOST ===

print("\n🚀 Entrenando XGBoost con WCE (scale_pos_weight)...")
start_time = time.time()

from xgboost import XGBClassifier

xgb_wce = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
)
xgb_wce.fit(X_train_features, y_train)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_xgb_wce = evaluate_model(xgb_wce, X_test_features, y_test, 'XGBoost + WCE')
metrics_xgb_wce['loss_type'] = 'WCE'
metrics_xgb_wce['train_time'] = round(elapsed, 2)
all_results.append(metrics_xgb_wce)

In [ ]:
# === WCE: LIGHTGBM ===

print("\n💡 Entrenando LightGBM con WCE (class_weight + scale_pos_weight)...")
start_time = time.time()

from lightgbm import LGBMClassifier

lgbm_wce = LGBMClassifier(
    n_estimators=600,
    max_depth=10,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)
lgbm_wce.fit(X_train_features, y_train)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_lgbm_wce = evaluate_model(lgbm_wce, X_test_features, y_test, 'LightGBM + WCE')
metrics_lgbm_wce['loss_type'] = 'WCE'
metrics_lgbm_wce['train_time'] = round(elapsed, 2)
all_results.append(metrics_lgbm_wce)

## Etapa 4: Entrenamiento con Focal Loss

**teoria.md**: *"Incorpora un factor de modulación que reduce la influencia de los ejemplos benignos fáciles de clasificar durante el descenso de gradiente, forzando al modelo a concentrar su presupuesto de aprendizaje en muestras difíciles."*

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Para modelos sklearn: usamos un enfoque de **2 pasos**:
1. Entrenar modelo base con `class_weight='balanced'`
2. Calcular `sample_weight` focal-inspired basado en predicciones del modelo base
3. Re-entrenar el modelo con los sample_weights focalizados

In [ ]:
# === FOCAL LOSS: CÁLCULO DE SAMPLE WEIGHTS ===

def compute_focal_sample_weights(y_true, y_pred_proba, alpha=0.75, gamma=2.0):
    """
    Calcula sample weights inspirados en Focal Loss para usar en modelos sklearn.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Los pesos son proporcionales a (1 - p_t)^gamma,
    dando más importancia a muestras difíciles de clasificar.
    
    Args:
        y_true: etiquetas reales
        y_pred_proba: probabilidades predichas para clase positiva
        alpha: peso para clase positiva (vulnerable)
        gamma: factor de enfoque (2.0-3.0 recomendado en teoria.md)
    """
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    
    # p_t: probabilidad del modelo para la clase correcta
    pt = np.where(y_true == 1, y_pred_proba, 1 - y_pred_proba)
    
    # Clamp para evitar log(0)
    pt = np.clip(pt, 1e-7, 1 - 1e-7)
    
    # Focal weight: (1 - p_t)^gamma
    focal_weight = (1 - pt) ** gamma
    
    # Alpha weight
    alpha_weight = np.where(y_true == 1, alpha, 1 - alpha)
    
    # Sample weight final
    sample_weights = alpha_weight * focal_weight
    
    # Normalizar para que el promedio sea 1
    sample_weights = sample_weights / sample_weights.mean()
    
    return sample_weights

print("✅ Función compute_focal_sample_weights definida")
print("   - alpha=0.75 (peso para clase vulnerable)")
print("   - gamma=2.0 (factor de enfoque, rango recomendado [2.0, 3.0])")

In [ ]:
# === FOCAL LOSS: RANDOM FOREST (2 pasos) ===

print("\n🌲 Focal Loss - Random Forest (entrenamiento en 2 pasos)...")

# Paso 1: Modelo base para obtener predicciones
print("   Paso 1: Entrenando modelo base...")
rf_base = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight='balanced',
    max_features='sqrt', n_jobs=-1, random_state=42
)
rf_base.fit(X_train_features, y_train)

# Obtener predicciones del modelo base sobre train
y_train_proba_base = rf_base.predict_proba(X_train_features)[:, 1]

# Paso 2: Calcular focal weights
print("   Paso 2: Calculando focal sample weights (gamma=2.0, alpha=0.75)...")
focal_weights = compute_focal_sample_weights(y_train, y_train_proba_base, alpha=0.75, gamma=2.0)
print(f"   Weight stats: min={focal_weights.min():.4f}, max={focal_weights.max():.4f}, mean={focal_weights.mean():.4f}")

# Paso 3: Re-entrenar con focal weights
print("   Paso 3: Re-entrenando con focal sample weights...")
start_time = time.time()

rf_focal = RandomForestClassifier(
    n_estimators=600, max_depth=None, min_samples_split=2,
    min_samples_leaf=1, max_features='sqrt', bootstrap=True,
    n_jobs=-1, random_state=42
)
rf_focal.fit(X_train_features, y_train, sample_weight=focal_weights)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_rf_focal = evaluate_model(rf_focal, X_test_features, y_test, 'RandomForest + Focal Loss')
metrics_rf_focal['loss_type'] = 'Focal'
metrics_rf_focal['train_time'] = round(elapsed, 2)
all_results.append(metrics_rf_focal)

In [ ]:
# === FOCAL LOSS: XGBOOST (2 pasos) ===

print("\n🚀 Focal Loss - XGBoost (entrenamiento en 2 pasos)...")

# Paso 1: Modelo base
print("   Paso 1: Entrenando XGBoost base...")
xgb_base = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', n_jobs=-1
)
xgb_base.fit(X_train_features, y_train)
y_train_proba_xgb = xgb_base.predict_proba(X_train_features)[:, 1]

# Paso 2: Focal weights
print("   Paso 2: Calculando focal weights (gamma=2.0)...")
focal_weights_xgb = compute_focal_sample_weights(y_train, y_train_proba_xgb, alpha=0.75, gamma=2.0)

# Paso 3: Re-entrenar
print("   Paso 3: Re-entrenando con focal weights...")
start_time = time.time()

xgb_focal = XGBClassifier(
    n_estimators=600, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_focal.fit(X_train_features, y_train, sample_weight=focal_weights_xgb)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_xgb_focal = evaluate_model(xgb_focal, X_test_features, y_test, 'XGBoost + Focal Loss')
metrics_xgb_focal['loss_type'] = 'Focal'
metrics_xgb_focal['train_time'] = round(elapsed, 2)
all_results.append(metrics_xgb_focal)

In [ ]:
# === FOCAL LOSS: LIGHTGBM (2 pasos) ===

print("\n💡 Focal Loss - LightGBM (entrenamiento en 2 pasos)...")

# Paso 1: Modelo base
print("   Paso 1: Entrenando LightGBM base...")
lgbm_base = LGBMClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42,
    verbose=-1, n_jobs=-1
)
lgbm_base.fit(X_train_features, y_train)
y_train_proba_lgbm = lgbm_base.predict_proba(X_train_features)[:, 1]

# Paso 2: Focal weights
print("   Paso 2: Calculando focal weights (gamma=2.0)...")
focal_weights_lgbm = compute_focal_sample_weights(y_train, y_train_proba_lgbm, alpha=0.75, gamma=2.0)

# Paso 3: Re-entrenar
print("   Paso 3: Re-entrenando con focal weights...")
start_time = time.time()

lgbm_focal = LGBMClassifier(
    n_estimators=600, max_depth=10, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
    random_state=42, verbose=-1, n_jobs=-1
)
lgbm_focal.fit(X_train_features, y_train, sample_weight=focal_weights_lgbm)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_lgbm_focal = evaluate_model(lgbm_focal, X_test_features, y_test, 'LightGBM + Focal Loss')
metrics_lgbm_focal['loss_type'] = 'Focal'
metrics_lgbm_focal['train_time'] = round(elapsed, 2)
all_results.append(metrics_lgbm_focal)

## Etapa 5: Entrenamiento con Class-Balanced Loss (CB)

**teoria.md**: *"Introduce un factor ponderador basado en el concepto del 'número efectivo de muestras'. El término de pérdida CB pondera la pérdida mediante un coeficiente que normaliza la diversidad intrínseca de los datos."*

$$\mathcal{L}_{\text{CB}} = \frac{1 - \beta}{1 - \beta^n} \mathcal{L}(p_t)$$

Donde β = (N-1)/N regula la tasa de saturación y n = población de la clase objetivo.

In [ ]:
# === CLASS-BALANCED LOSS: CÁLCULO DE SAMPLE WEIGHTS ===

def compute_cb_sample_weights(y_true, beta=0.9999):
    """
    Calcula sample weights usando la fórmula Class-Balanced de teoria.md.
    
    effective_num = (1 - beta^n) / (1 - beta)
    weight = 1 / effective_num
    
    Args:
        y_true: etiquetas reales
        beta: factor de saturación. beta = (N-1)/N según teoria.md
    """
    y_true = np.array(y_true)
    
    n_vuln = np.sum(y_true == 1)
    n_safe = np.sum(y_true == 0)
    
    # Número efectivo de muestras
    effective_num_vuln = (1 - beta ** n_vuln) / (1 - beta) if n_vuln > 0 else 1.0
    effective_num_safe = (1 - beta ** n_safe) / (1 - beta) if n_safe > 0 else 1.0
    
    # Pesos inversamente proporcionales al número efectivo
    weight_vuln = 1.0 / effective_num_vuln
    weight_safe = 1.0 / effective_num_safe
    
    # Normalizar para que el promedio sea 1
    total = weight_vuln * n_vuln + weight_safe * n_safe
    weight_vuln = weight_vuln * len(y_true) / total
    weight_safe = weight_safe * len(y_true) / total
    
    # Asignar pesos por muestra
    sample_weights = np.where(y_true == 1, weight_vuln, weight_safe)
    
    print(f"   CB weights: vuln={weight_vuln:.4f}, safe={weight_safe:.4f}")
    print(f"   Effective num: vuln={effective_num_vuln:.2f}, safe={effective_num_safe:.2f}")
    print(f"   Ratio CB weight: {weight_vuln/weight_safe:.4f}x")
    
    return sample_weights

print("✅ Función compute_cb_sample_weights definida")
print("   - beta = 0.9999 (factor de saturación de información sintáctica)")

In [ ]:
# === CB LOSS: RANDOM FOREST ===

print("\n🌲 Class-Balanced Loss - Random Forest...")
cb_weights_rf = compute_cb_sample_weights(y_train, beta=0.9999)

start_time = time.time()
rf_cb = RandomForestClassifier(
    n_estimators=600, max_depth=None, min_samples_split=2,
    min_samples_leaf=1, max_features='sqrt', bootstrap=True,
    n_jobs=-1, random_state=42
)
rf_cb.fit(X_train_features, y_train, sample_weight=cb_weights_rf)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_rf_cb = evaluate_model(rf_cb, X_test_features, y_test, 'RandomForest + CB Loss')
metrics_rf_cb['loss_type'] = 'CB'
metrics_rf_cb['train_time'] = round(elapsed, 2)
all_results.append(metrics_rf_cb)

In [ ]:
# === CB LOSS: XGBOOST ===

print("\n🚀 Class-Balanced Loss - XGBoost...")
cb_weights_xgb = compute_cb_sample_weights(y_train, beta=0.9999)

start_time = time.time()
xgb_cb = XGBClassifier(
    n_estimators=600, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_cb.fit(X_train_features, y_train, sample_weight=cb_weights_xgb)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_xgb_cb = evaluate_model(xgb_cb, X_test_features, y_test, 'XGBoost + CB Loss')
metrics_xgb_cb['loss_type'] = 'CB'
metrics_xgb_cb['train_time'] = round(elapsed, 2)
all_results.append(metrics_xgb_cb)

In [ ]:
# === CB LOSS: LIGHTGBM ===

print("\n💡 Class-Balanced Loss - LightGBM...")
cb_weights_lgbm = compute_cb_sample_weights(y_train, beta=0.9999)

start_time = time.time()
lgbm_cb = LGBMClassifier(
    n_estimators=600, max_depth=10, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
    random_state=42, verbose=-1, n_jobs=-1
)
lgbm_cb.fit(X_train_features, y_train, sample_weight=cb_weights_lgbm)
elapsed = time.time() - start_time
print(f"⏱️ Entrenamiento completado en {elapsed:.1f}s")

metrics_lgbm_cb = evaluate_model(lgbm_cb, X_test_features, y_test, 'LightGBM + CB Loss')
metrics_lgbm_cb['loss_type'] = 'CB'
metrics_lgbm_cb['train_time'] = round(elapsed, 2)
all_results.append(metrics_lgbm_cb)

## Etapa 6: Comparación de Resultados

Comparación exhaustiva de **todas las combinaciones** (modelo × función de pérdida).

In [ ]:
# === TABLA COMPARATIVA ===

df_results = pd.DataFrame(all_results)
display_cols = ['model', 'loss_type', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'train_time']
df_display = df_results[display_cols].sort_values('f1_score', ascending=False)

print("\n" + "="*100)
print("  📊 TABLA COMPARATIVA DE RESULTADOS (ordenado por F1 Score)")
print("="*100)
print(df_display.to_string(index=False))
print("="*100)

# Mejor modelo
best = df_display.iloc[0]
print(f"\n🏆 MEJOR MODELO: {best['model']}")
print(f"   F1 Score: {best['f1_score']:.4f}")
print(f"   Recall:   {best['recall']:.4f}")
print(f"   ROC AUC:  {best['roc_auc']:.4f}")

# Comparar con métricas anteriores
print(f"\n📈 MEJORA vs. modelo anterior (accuracy=0.56, F1=0.5926):")
print(f"   Accuracy: {best['accuracy']:.4f} vs 0.5600 ({'+' if best['accuracy'] > 0.56 else ''}{(best['accuracy']-0.56)*100:.2f}pp)")
print(f"   F1 Score: {best['f1_score']:.4f} vs 0.5926 ({'+' if best['f1_score'] > 0.5926 else ''}{(best['f1_score']-0.5926)*100:.2f}pp)")

In [ ]:
# === GRÁFICOS COMPARATIVOS ===

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Colores por loss type
loss_colors = {'WCE': '#3498db', 'Focal': '#e74c3c', 'CB': '#2ecc71'}
model_short = {'RandomForest': 'RF', 'XGBoost': 'XGB', 'LightGBM': 'LGBM'}

# Preparar datos para gráficos
models_list = df_results['model'].tolist()
colors = [loss_colors.get(lt, '#95a5a6') for lt in df_results['loss_type']]

# 1. Barras de F1 Score
ax = axes[0, 0]
bars = ax.barh(range(len(models_list)), df_results['f1_score'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(models_list)))
ax.set_yticklabels(models_list, fontsize=9)
ax.set_xlabel('F1 Score')
ax.set_title('F1 Score por Modelo + Loss Function', fontweight='bold')
ax.axvline(x=0.5926, color='gray', linestyle='--', label='Anterior (0.5926)')
ax.legend(fontsize=8)
for i, v in enumerate(df_results['f1_score'].values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)

# 2. Barras de Recall (crítico para detección de vulnerabilidades)
ax = axes[0, 1]
bars = ax.barh(range(len(models_list)), df_results['recall'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(models_list)))
ax.set_yticklabels(models_list, fontsize=9)
ax.set_xlabel('Recall')
ax.set_title('Recall por Modelo + Loss Function (detectar vulnerabilidades)', fontweight='bold')
ax.axvline(x=0.5882, color='gray', linestyle='--', label='Anterior (0.5882)')
ax.legend(fontsize=8)
for i, v in enumerate(df_results['recall'].values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)

# 3. Precision vs Recall scatter
ax = axes[1, 0]
for i, row in df_results.iterrows():
    color = loss_colors.get(row['loss_type'], '#95a5a6')
    marker = {'WCE': 'o', 'Focal': 's', 'CB': '^'}.get(row['loss_type'], 'o')
    ax.scatter(row['recall'], row['precision'], c=color, marker=marker, s=120, zorder=5,
               label=f"{row['model']}" if i < 3 else None)
    ax.annotate(row['model'].split(' + ')[0][:4], (row['recall'], row['precision']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall', fontweight='bold')
ax.grid(alpha=0.3)
# Legend for loss types
for lt, c in loss_colors.items():
    ax.scatter([], [], c=c, label=lt, s=80)
ax.legend(fontsize=8)

# 4. ROC AUC comparison
ax = axes[1, 1]
bars = ax.barh(range(len(models_list)), df_results['roc_auc'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(models_list)))
ax.set_yticklabels(models_list, fontsize=9)
ax.set_xlabel('ROC AUC')
ax.set_title('ROC AUC por Modelo + Loss Function', fontweight='bold')
ax.axvline(x=0.5684, color='gray', linestyle='--', label='Anterior (0.5684)')
ax.legend(fontsize=8)
for i, v in enumerate(df_results['roc_auc'].values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Gráfico guardado en reports/model_comparison.png")

In [ ]:
# === MATRICES DE CONFUSIÓN COMPARATIVAS ===

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle('Matrices de Confusión: Todos los Modelos × Loss Functions', fontsize=14, fontweight='bold')

for idx, result in enumerate(all_results):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    cm = np.array(result['confusion_matrix'])
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f"{result['model']}\nF1={result['f1_score']:.4f}", fontsize=10)
    
    # Anotar celdas
    for i in range(2):
        for j in range(2):
            text = ax.text(j, i, f'{cm[i, j]}',
                          ha='center', va='center', fontsize=12,
                          color='white' if cm[i, j] > cm.max()/2 else 'black')
    
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Seguro', 'Vuln'])
    ax.set_yticklabels(['Seguro', 'Vuln'])

plt.tight_layout()
plt.savefig('reports/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Matrices de confusión guardadas en reports/confusion_matrices.png")

In [ ]:
# === CURVAS ROC SUPERPUESTAS ===

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Lista de modelos entrenados y sus nombres
trained_models = [
    (rf_wce, 'RF + WCE'),
    (xgb_wce, 'XGB + WCE'),
    (lgbm_wce, 'LGBM + WCE'),
    (rf_focal, 'RF + Focal'),
    (xgb_focal, 'XGB + Focal'),
    (lgbm_focal, 'LGBM + Focal'),
    (rf_cb, 'RF + CB'),
    (xgb_cb, 'XGB + CB'),
    (lgbm_cb, 'LGBM + CB'),
]

line_styles = ['-', '--', '-.', '-', '--', '-.', '-', '--', '-.']
color_map = plt.cm.tab10(np.linspace(0, 1, len(trained_models)))

for i, (model, name) in enumerate(trained_models):
    y_proba = model.predict_proba(X_test_features)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})',
            color=color_map[i], linestyle=line_styles[i], linewidth=1.5)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC=0.50)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curvas ROC — Comparación de Todos los Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Curvas ROC guardadas en reports/roc_curves.png")

## Etapa 7: Entrenamiento Final con Todos los Datos

Tomamos la **mejor combinación** (modelo + loss function) y re-entrenamos con:
1. Todos los datos disponibles (sin limit_per_source)
2. Cross-validation completo de 5 folds
3. Guardar modelo final en `models/vulnerability_model.joblib`

In [ ]:
# === SELECCIONAR MEJOR MODELO ===

best_result = df_results.sort_values('f1_score', ascending=False).iloc[0]
best_model_name = best_result['model']
best_loss_type = best_result['loss_type']

print(f"🏆 Mejor modelo seleccionado: {best_model_name}")
print(f"   Loss Function: {best_loss_type}")
print(f"   F1 Score (prueba): {best_result['f1_score']:.4f}")
print(f"   Recall (prueba): {best_result['recall']:.4f}")
print(f"   ROC AUC (prueba): {best_result['roc_auc']:.4f}")

print("\n⚠️ Para el entrenamiento final, se recargarán TODOS los datos sin límite.")
print("   Esto puede tomar varios minutos dependiendo del tamaño total.")

In [ ]:
# === CARGAR DATOS COMPLETOS (SIN LÍMITE POR FUENTE) ===
# NOTA: Para datasets muy grandes (MegaVul >1GB), cargamos un límite razonable
# para evitar problemas de memoria. Ajustar según RAM disponible.

FULL_LIMIT = 50000  # Límite por fuente para entrenamiento final (ajustar según RAM)

print("\n📦 Cargando datasets COMPLETOS para entrenamiento final...")
all_datasets_full = {}

# CodeXGLUE (todo)
for name, path in codexglue_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='func', label_field='target', limit=FULL_LIMIT)
        all_datasets_full[name] = data
        print(f"  ✅ {name}: {len(data):,} muestras")

# D2A (todo)
for name, path in d2a_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='functionSource', label_field='label', limit=FULL_LIMIT)
        all_datasets_full[name] = data
        print(f"  ✅ {name}: {len(data):,} muestras")

# ReVeal (todo)
for name, path in reveal_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='code', label_field='label', limit=FULL_LIMIT)
        all_datasets_full[name] = data
        print(f"  ✅ {name}: {len(data):,} muestras")

# MegaVul C/C++
if megavul_cpp_path.exists():
    data = load_megavul(megavul_cpp_path, limit=FULL_LIMIT)
    all_datasets_full['megavul_cpp'] = data
    print(f"  ✅ megavul_cpp: {len(data):,} muestras")

# MegaVul Java
if megavul_java_path.exists():
    data = load_megavul(megavul_java_path, limit=FULL_LIMIT)
    all_datasets_full['megavul_java'] = data
    print(f"  ✅ megavul_java: {len(data):,} muestras")

# CVEFixes
if cvefixes_path.exists():
    data = load_cvefixes_csv(cvefixes_path, limit=FULL_LIMIT)
    all_datasets_full['cvefixes'] = data
    print(f"  ✅ cvefixes: {len(data):,} muestras")

# OWASP
for name, path in owasp_files:
    p = PROJECT_ROOT / path
    if p.exists():
        data = load_jsonl(p, code_field='raw_code', label_field='is_vulnerable')
        all_datasets_full[name] = data
        print(f"  ✅ {name}: {len(data):,} muestras")

# Combinar
combined_full = []
for source_name, samples in all_datasets_full.items():
    for s in samples:
        s_copy = s.copy()
        s_copy['source'] = source_name
        combined_full.append(s_copy)

random.shuffle(combined_full)

df_full = pd.DataFrame(combined_full)
X_full = df_full['raw_code'].values
y_full = df_full['is_vulnerable'].values

total_full = len(combined_full)
vuln_full = np.sum(y_full == 1)
print(f"\n{'='*70}")
print(f"📊 DATASET FINAL: {total_full:,} muestras")
print(f"   Vulnerables: {vuln_full:,} ({100*vuln_full/total_full:.1f}%)")
print(f"   Seguras: {total_full-vuln_full:,} ({100*(total_full-vuln_full)/total_full:.1f}%)")
print(f"{'='*70}")

In [ ]:
# === ENTRENAMIENTO FINAL ===

print(f"\n🏋️ Entrenamiento final: {best_model_name}")
print(f"   Dataset total: {total_full:,} muestras")

# Split
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print(f"   Train: {len(X_train_full):,} | Test: {len(X_test_full):,}")

# Feature engineering
print("\n🔧 Construyendo features sobre dataset completo...")
start_time = time.time()

feature_pipeline_final = FeatureUnion([
    ('tfidf', TfidfVectorizer(
        max_features=5000, ngram_range=(1, 3), sublinear_tf=True,
        min_df=2, max_df=0.95, strip_accents='unicode', token_pattern=r'\b\w+\b'
    )),
    ('ast', RobustASTFeatureExtractor()),
    ('security', SecurityFeatureExtractor()),
    ('language', LanguageFeatureExtractor()),
])

X_train_feat_full = feature_pipeline_final.fit_transform(X_train_full)
X_test_feat_full = feature_pipeline_final.transform(X_test_full)
elapsed_feat = time.time() - start_time
print(f"   ✅ Features shape: {X_train_feat_full.shape}")
print(f"   ⏱️ Feature engineering: {elapsed_feat:.1f}s")

# Calcular pesos según la mejor loss function
n_vuln_full = np.sum(y_train_full == 1)
n_safe_full = np.sum(y_train_full == 0)
w1_full = len(y_train_full) / (2 * n_vuln_full)
w0_full = len(y_train_full) / (2 * n_safe_full)
spw_full = n_safe_full / n_vuln_full

# Crear modelo final según el mejor encontrado
print(f"\n🏋️ Entrenando modelo final ({best_model_name})...")
start_time = time.time()

if 'RandomForest' in best_model_name:
    final_model = RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features='sqrt', bootstrap=True,
        n_jobs=-1, random_state=42
    )
elif 'XGBoost' in best_model_name:
    final_model = XGBClassifier(
        n_estimators=600, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        random_state=42, eval_metric='logloss', n_jobs=-1
    )
elif 'LightGBM' in best_model_name:
    final_model = LGBMClassifier(
        n_estimators=600, max_depth=10, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
        random_state=42, verbose=-1, n_jobs=-1
    )

# Aplicar la mejor loss function
if best_loss_type == 'WCE':
    if 'RandomForest' in best_model_name:
        final_model.set_params(class_weight={0: w0_full, 1: w1_full})
        final_model.fit(X_train_feat_full, y_train_full)
    else:
        final_model.set_params(scale_pos_weight=spw_full)
        final_model.fit(X_train_feat_full, y_train_full)
elif best_loss_type == 'Focal':
    # 2-step focal
    print("   Paso 1: Modelo base para focal weights...")
    if 'RandomForest' in best_model_name:
        base_model = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                            max_features='sqrt', n_jobs=-1, random_state=42)
    elif 'XGBoost' in best_model_name:
        base_model = XGBClassifier(n_estimators=300, scale_pos_weight=spw_full,
                                   random_state=42, eval_metric='logloss', n_jobs=-1)
    else:
        base_model = LGBMClassifier(n_estimators=300, scale_pos_weight=spw_full,
                                    random_state=42, verbose=-1, n_jobs=-1)
    base_model.fit(X_train_feat_full, y_train_full)
    y_proba_base = base_model.predict_proba(X_train_feat_full)[:, 1]
    focal_w = compute_focal_sample_weights(y_train_full, y_proba_base, alpha=0.75, gamma=2.0)
    print("   Paso 2: Entrenando con focal weights...")
    final_model.fit(X_train_feat_full, y_train_full, sample_weight=focal_w)
elif best_loss_type == 'CB':
    cb_w = compute_cb_sample_weights(y_train_full, beta=0.9999)
    final_model.fit(X_train_feat_full, y_train_full, sample_weight=cb_w)

elapsed_train = time.time() - start_time
print(f"   ⏱️ Entrenamiento: {elapsed_train:.1f}s")

# Evaluar
metrics_final = evaluate_model(final_model, X_test_feat_full, y_test_full, f'FINAL: {best_model_name}')

# Cross-validation
print("\n🔄 Cross-validation (5 folds)...")
# Para CV, necesitamos re-fit features; usamos el pipeline completo
# Debido al costo computacional, hacemos CV con un subset si es muy grande
cv_limit = min(20000, len(X_full))
X_cv = X_full[:cv_limit]
y_cv = y_full[:cv_limit]

cv_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('tfidf', TfidfVectorizer(
            max_features=5000, ngram_range=(1, 3), sublinear_tf=True,
            min_df=2, max_df=0.95, strip_accents='unicode', token_pattern=r'\b\w+\b'
        )),
        ('security', SecurityFeatureExtractor()),
        ('language', LanguageFeatureExtractor()),
    ])),
    ('clf', final_model)
])

cv_scores = cross_val_score(cv_pipeline, X_cv, y_cv, cv=5, scoring='f1', n_jobs=-1)
print(f"   CV F1 scores: {cv_scores}")
print(f"   CV F1 mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# === GUARDAR MODELO FINAL ===

import joblib

# Crear pipeline completo (features + modelo)
final_pipeline = Pipeline([
    ('features', feature_pipeline_final),
    ('clf', final_model)
])

# Re-fit sobre TODOS los datos
print("\n🏋️ Fit final sobre TODO el dataset...")
final_pipeline.fit(X_full, y_full)

# Guardar
model_path = PROJECT_ROOT / 'models' / 'vulnerability_model.joblib'
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, model_path)
print(f"\n✅ Modelo guardado en: {model_path}")
print(f"   Tamaño: {model_path.stat().st_size / (1024*1024):.1f} MB")

# Guardar reporte de métricas
report_path = PROJECT_ROOT / 'reports' / 'metrics.json'
metrics_final['cross_validation_f1_mean'] = round(float(cv_scores.mean()), 4)
metrics_final['cross_validation_f1_std'] = round(float(cv_scores.std()), 4)
metrics_final['dataset_total_samples'] = total_full
metrics_final['dataset_sources'] = list(all_datasets_full.keys())
metrics_final['loss_function'] = best_loss_type
metrics_final['technique'] = f'teoria.md: {best_loss_type} ({best_model_name})'

report_path.write_text(json.dumps(metrics_final, indent=2), encoding='utf-8')
print(f"✅ Reporte guardado en: {report_path}")

# Guardar todos los resultados comparativos
all_results_path = PROJECT_ROOT / 'reports' / 'all_training_results.json'
all_results_path.write_text(json.dumps(all_results, indent=2), encoding='utf-8')
print(f"✅ Resultados comparativos guardados en: {all_results_path}")

In [ ]:
# === RESUMEN FINAL ===

print("\n" + "="*80)
print("  🏆 RESUMEN FINAL DEL ENTRENAMIENTO AVANZADO")
print("="*80)

print(f"\n📊 Técnicas de teoria.md implementadas:")
print(f"   1. ✅ Weighted Cross-Entropy (WCE) - w1={w1:.4f}, w0={w0:.4f}")
print(f"   2. ✅ Focal Loss (FL) - gamma=2.0, alpha=0.75")
print(f"   3. ✅ Class-Balanced Loss (CB) - beta=0.9999")

print(f"\n📦 Datasets utilizados:")
for name, data in all_datasets_full.items():
    vuln = sum(1 for d in data if d['is_vulnerable'])
    print(f"   - {name}: {len(data):,} muestras (vuln={vuln:,})")

print(f"\n🤖 Modelos evaluados (prueba con subset):")
for r in sorted(all_results, key=lambda x: x['f1_score'], reverse=True):
    print(f"   {r['model']:35s} | F1={r['f1_score']:.4f} | Recall={r['recall']:.4f} | AUC={r['roc_auc']:.4f}")

print(f"\n🏆 Mejor modelo: {best_model_name}")
print(f"   F1 Score final: {metrics_final['f1_score']:.4f}")
print(f"   Recall final:   {metrics_final['recall']:.4f}")
print(f"   ROC AUC final:  {metrics_final['roc_auc']:.4f}")
print(f"   CV F1 mean:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n📈 MEJORA vs. modelo anterior:")
print(f"   Accuracy: {metrics_final['accuracy']:.4f} vs 0.5600 ({'+' if metrics_final['accuracy'] > 0.56 else ''}{(metrics_final['accuracy']-0.56)*100:.2f}pp)")
print(f"   F1 Score: {metrics_final['f1_score']:.4f} vs 0.5926 ({'+' if metrics_final['f1_score'] > 0.5926 else ''}{(metrics_final['f1_score']-0.5926)*100:.2f}pp)")
print(f"   ROC AUC:  {metrics_final['roc_auc']:.4f} vs 0.5684 ({'+' if metrics_final['roc_auc'] > 0.5684 else ''}{(metrics_final['roc_auc']-0.5684)*100:.2f}pp)")

print(f"\n📁 Archivos generados:")
print(f"   - models/vulnerability_model.joblib")
print(f"   - reports/metrics.json")
print(f"   - reports/all_training_results.json")
print(f"   - reports/class_distribution.png")
print(f"   - reports/model_comparison.png")
print(f"   - reports/confusion_matrices.png")
print(f"   - reports/roc_curves.png")
print("="*80)
print("\n✅ ¡Entrenamiento avanzado completado exitosamente!")